In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# 1. Cấu hình đường dẫn (Path config)
# Giả định bạn chạy file này từ thư mục gốc của project BountyHunter
DATA_RAW_DIR = Path("data/raw")
REPORTS_DIR = Path("reports/figures")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

def generate_chart_1a():
    print("🚀 Đang xử lý dữ liệu...")
    
    # 2. Load dữ liệu từ CSV
    try:
        orders = pd.read_csv(DATA_RAW_DIR / "orders.csv")
        order_items = pd.read_csv(DATA_RAW_DIR / "order_items.csv")
    except FileNotFoundError:
        print("❌ Lỗi: Không tìm thấy file trong data/raw/. Vui lòng kiểm tra lại đường dẫn.")
        return

    # 3. Tiền xử lý (Pre-processing)
    orders['order_date'] = pd.to_datetime(orders['order_date'])
    
    # Lọc các đơn hàng đóng góp vào Revenue tuyệt đối
    # Loại bỏ 'cancelled' và 'returned'
    valid_statuses = ['delivered', 'shipped', 'paid']
    success_orders = orders[orders['order_status'].isin(valid_statuses)]
    
    # Merge để lấy thông tin giá và lượng
    df = pd.merge(success_orders, order_items, on='order_id')
    
    # Tính Revenue = Quantity * Unit_Price
    df['line_revenue'] = df['quantity'] * df['unit_price']
    df['year'] = df['order_date'].dt.year
    
    # 4. Tính toán Metrics theo năm
    yearly_stats = df.groupby('year')['line_revenue'].sum().reset_index()
    yearly_stats.columns = ['year', 'revenue']
    yearly_stats = yearly_stats.sort_values('year')
    
    # Tính YoY Growth Rate
    yearly_stats['yoy_growth'] = yearly_stats['revenue'].pct_change() * 100

    # 5. Vẽ biểu đồ Dual-axis
    print("📊 Đang khởi tạo biểu đồ...")
    fig, ax1 = plt.subplots(figsize=(14, 8))
    sns.set_style("whitegrid")

    # Trục trái: Bar chart cho Revenue
    color_rev = '#2E7D32' 
    ax1.set_xlabel('Năm', fontsize=12)
    ax1.set_ylabel('Doanh thu tuyệt đối (VNĐ)', color=color_rev, fontsize=12, fontweight='bold')
    bars = ax1.bar(yearly_stats['year'], yearly_stats['revenue'], color=color_rev, alpha=0.6, label='Revenue')
    ax1.tick_params(axis='y', labelcolor=color_rev)

    # Thêm nhãn doanh thu (đơn vị tỷ VNĐ) trên đầu cột
    for bar in bars:
        height = bar.get_height()
        ax1.annotate(f'{height/1e9:.2f}B',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 5), textcoords="offset points",
                    ha='center', va='bottom', fontsize=10, fontweight='bold')

    # Trục phải: Line chart cho YoY%
    ax2 = ax1.twinx()
    color_yoy = '#D32F2F'
    ax2.set_ylabel('YoY Growth Rate (%)', color=color_yoy, fontsize=12, fontweight='bold')
    line = ax2.plot(yearly_stats['year'], yearly_stats['yoy_growth'], color=color_yoy, 
                    marker='o', linewidth=3, markersize=8, label='YoY % Growth')
    ax2.tick_params(axis='y', labelcolor=color_yoy)

    # Thêm nhãn % tăng trưởng trên đường line
    for i, row in yearly_stats.iterrows():
        if pd.notnull(row['yoy_growth']):
            ax2.annotate(f"{row['yoy_growth']:.1f}%", 
                         (row['year'], row['yoy_growth']),
                         textcoords="offset points", xytext=(0, 12), 
                         ha='center', color=color_yoy, fontweight='bold')

    # Đường mốc 0% để thấy rõ tăng trưởng âm
    ax2.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)

    plt.title('CHART 1a: REVENUE TREND & YoY GROWTH RATE (2012-2022)', fontsize=16, fontweight='bold', pad=25)
    
    # Lưu kết quả
    output_file = REPORTS_DIR / "chart_1a_revenue_trend.png"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✅ Hoàn thành! Biểu đồ đã được lưu tại: {output_file}")

if __name__ == "__main__":
    generate_chart_1a()

🚀 Đang xử lý dữ liệu...
❌ Lỗi: Không tìm thấy file trong data/raw/. Vui lòng kiểm tra lại đường dẫn.
